In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 클래식 피드포워드와 제어 흐름 (동적 Circuit)

<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
동적 Circuit은 강력한 도구로, 양자 Circuit 실행 중에 Qubit을 측정하고 해당 중간 Circuit 측정 결과를 바탕으로 Circuit 내에서 클래식 논리 연산을 수행할 수 있습니다. 이 과정은 _클래식 피드포워드_라고도 알려져 있습니다. 동적 Circuit을 어떻게 활용할지에 대한 이해는 아직 초기 단계이지만, 양자 연구 커뮤니티는 이미 다음과 같은 여러 활용 사례를 발굴했습니다.

* [GHZ 상태](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [W-상태](https://arxiv.org/abs/2403.07604) 등 효율적인 양자 상태 준비 (W-상태에 대한 자세한 내용은 ["피드포워드를 사용하는 얕은 Circuit에 의한 상태 준비"](https://arxiv.org/abs/2307.14840)도 참조하세요), 및 광범위한 [행렬 곱 상태](https://arxiv.org/abs/2404.16083)
* 얕은 Circuit을 이용한 같은 칩 위의 qubit 간 [효율적인 장거리 얽힘](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)
* [IQP 유사 Circuit의 효율적인 샘플링](https://arxiv.org/pdf/2505.04705)
## `if` 문
`if` 문은 클래식 비트 또는 레지스터의 값을 기반으로 조건부로 연산을 수행하는 데 사용됩니다.

아래 예제에서는 Qubit에 Hadamard Gate를 적용하고 측정합니다. 결과가 1이면 Qubit에 X Gate를 적용하여 0 상태로 되돌립니다. 그런 다음 Qubit을 다시 측정합니다. 최종 측정 결과는 100% 확률로 0이 됩니다.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

`with` 문에는 컨텍스트 매니저 자체인 할당 대상을 지정할 수 있으며, 이를 저장해 두었다가 나중에 else 블록을 만드는 데 사용할 수 있습니다. else 블록은 `if` 블록의 내용이 *실행되지 않을* 때마다 실행됩니다.

아래 예제에서는 두 Qubit과 두 클래식 비트로 레지스터를 초기화합니다. 첫 번째 Qubit에 Hadamard Gate를 적용하고 측정합니다. 결과가 1이면 두 번째 Qubit에 Hadamard Gate를 적용하고, 그렇지 않으면 두 번째 Qubit에 X Gate를 적용합니다. 마지막으로 두 번째 Qubit도 측정합니다.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

단일 클래식 비트에 조건을 거는 것 외에도, 여러 비트로 구성된 클래식 레지스터의 값에 조건을 거는 것도 가능합니다.

아래 예제에서는 두 Qubit에 Hadamard Gate를 적용하고 측정합니다. 결과가 `01`, 즉 첫 번째 Qubit이 1이고 두 번째 Qubit이 0이면, 세 번째 Qubit에 X Gate를 적용합니다. 마지막으로 세 번째 Qubit을 측정합니다. 명확성을 위해 세 번째 클래식 비트의 상태(0)를 `if` 조건에 명시적으로 지정했습니다. Circuit 다이어그램에서 조건은 조건이 걸린 클래식 비트의 원으로 표시됩니다. 실선 원은 1에 대한 조건을, 테두리만 있는 원은 0에 대한 조건을 나타냅니다.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## 클래식 표현식
Qiskit 클래식 표현식 모듈 [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical)은 Circuit 실행 중 클래식 값에 대한 런타임 연산의 탐색적 표현을 포함합니다. 하드웨어 제한으로 인해 현재는 `QuantumCircuit.if_test()` 조건만 지원됩니다.

다음 예제는 패리티 계산을 사용하여 동적 Circuit으로 n-Qubit GHZ 상태를 생성하는 방법을 보여줍니다. 먼저 인접한 Qubit에 $n/2$개의 Bell 쌍을 생성합니다. 그런 다음 쌍 사이에 CNOT Gate 레이어를 사용하여 이 쌍들을 연결합니다. 이전 CNOT Gate의 타겟 Qubit을 모두 측정하고 측정된 각 Qubit을 $\vert 0 \rangle$ 상태로 초기화합니다. 이전 모든 비트의 패리티가 홀수인 측정되지 않은 각 사이트에 $X$를 적용합니다. 마지막으로 측정된 Qubit에 CNOT Gate를 적용하여 측정에서 손실된 얽힘을 재확립합니다.

패리티 계산에서 구성된 표현식의 첫 번째 요소는 Python 객체 `mr[0]`을 [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) 노드로 들어올리는 것입니다 (`lift`는 임의의 객체를 클래식 표현식으로 변환하는 데 사용됩니다). `mr[1]`과 이후의 가능한 클래식 레지스터에는 이것이 필요하지 않습니다. 이들은 `expr.bit_xor`의 입력이며, 필요한 변환은 이 경우에 자동으로 수행됩니다. 이러한 표현식은 루프 및 다른 구조에서 점차적으로 구성할 수 있습니다.

In [4]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:
    with case(0):
        circuit.x(q0)
    with case(1):
        circuit.z(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif" alt="Output of the previous code cell" />

Because the example above used a single classical bit, there were only two possible cases, so you could have achieved the same result using an if-else statement. The switch case is mainly useful when branching on the value of a classical register composed of multiple bits. The following example shows how to construct a default case, which is executed if none of the preceding cases are. Note that in a switch statement, only one of the blocks are ever executed. There is no fallthrough.

The example below applies Hadamard gates to two qubits and measures them. If the result is either 00 or 11, apply a Z gate to the third qubit. If the result is 01, apply a Y gate. If none of the preceding cases matched, apply an X gate. Finally, measure the third qubit.

In [5]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:
    with case(0b000, 0b011):
        circuit.z(q2)
    with case(0b001):
        circuit.y(q2)
    with case(case.DEFAULT):
        circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

<span id="store"></span>
### `Store`

클래식 표현식이 반복적으로 사용될 경우, [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) 명령을 사용하여 해당 표현식의 결과를 저장할 수 있습니다. 연산은 자동으로 병렬화되어 런타임 효율성이 크게 향상됩니다.

예를 들어, $B = \neg A$인 경우 $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$보다 $B[0] \oplus B[1] \oplus B[2] \ldots$로 표현하는 것이 더 자연스럽고 런타임에서도 더 효율적입니다. 전자는 XOR 체인 전에 단일 병렬 단계에서 부정을 계산하는 반면, 후자는 표현식 내에서 각 부정을 순차적으로 평가합니다.

전체 예제:

In [6]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:
    circuit.x(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/e64ec241-41e8-40f8-ab64-af236c6c7802-0.avif)

## 다음 단계
> **Tip:** - [stretch](/guides/stretch)를 사용하여 정확한 동적 디커플링을 구현하는 방법을 알아보세요.
> - [Circuit 스케줄 시각화](/guides/qiskit-runtime-circuit-timing)를 사용하여 동적 Circuit을 디버그하고 최적화하세요.
> - [동적 Circuit 실행](/guides/execute-dynamic-circuits).

In [7]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):
    circuit.h([q0, q1])
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 334, '10': 368, '00': 322}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif" alt="Output of the previous code cell" />

## Classical expressions

The Qiskit classical expression module [`qiskit.circuit.classical`](/docs/api/qiskit/circuit_classical) contains an exploratory representation of runtime operations on classical values during circuit execution.

The following example shows that you can use the calculation of the parity to create an n-qubit GHZ state using dynamic circuits. First, generate $n/2$ Bell pairs on adjacent qubits. Then, glue these pairs together using a layer of CNOT gates in between pairs. You then measure the target qubit of all prior CNOT gates and reset each measured qubit to the state $\vert 0 \rangle$. You apply $X$ to every unmeasured site for which the parity of all preceding bits is odd. Finally, CNOT gates are applied to the measured qubits to re-establish the entanglement lost in the measurement.


In the parity calculation, the first element of the constructed expression involves lifting the Python object `mr[0]` to a [`Value`](/docs/api/qiskit/circuit_classical#value) node (`lift` is used to turn arbitrary objects into classical expressions). This is not necessary for `mr[1]` and the possible following classical register, as they are inputs to `expr.bit_xor`, and any necessary lifting is done automatically in these cases. Such expressions can be built up in loops and other constructs.

In [8]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [9]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif" alt="Output of the previous code cell" />

<span id="store"></span>
### `Store`

You can use the [`store`](/docs/api/qiskit/circuit#store) instruction to save the result of a classical expression, if that expression will be used repeatedly. Operations are automatically parallelized, making your code significantly more efficient at runtime.

For example, it is more natural and more efficient at runtime to write $B[0] \oplus B[1] \oplus B[2] \ldots$, where $B = \neg A$, than $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  The former computes the negation in a single parallel step before the XOR chain, instead of evaluating each negation sequentially inside the expression.

Full example:

In [10]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
- [Execute dynamic circuits](/docs/guides/execute-dynamic-circuits).
</Admonition>